# 0. Global settings and shared functions

Every other notebook starts with `%run ./00_settings.ipynb`, which runs this notebook. It defines:

1. **where everything lives** (the master folder and the folders below it): the only cell you need to edit;
2. the **fixed constants** of the analysis: race and ethnicity groups, diagnoses, models, seeds and regularisation grids;
3. the **shared functions**: data loaders, logistic-regression probes (GPU and CPU), patient-level cross-fitting, the bootstrap, the two error measures (flagging rate and balanced accuracy), least-squares concept erasure (LEACE), matching, and the plot style.

**Data use.** EchoNext and the MIMIC-IV modules are distributed by PhysioNet under data use agreements that do not allow individual-level data to be shared.
Every notebook therefore prints and saves only aggregate values (counts, AUROCs, differences between groups) under `RESULTS_DIR` and `OUTPUT_DIR`.
Row-level files (cohort tables, waveforms, embeddings, predictions) are written to `WORK_DIR` only. Never commit `DATA_DIR`, `MODEL_DIR`, `WORK_DIR` or notebook outputs to a public repository.

## 0.1 Global settings (edit this cell)

`MASTER_DIR` is the folder below which everything is read and written. Raw data are expected in the folders that the PhysioNet download command creates (see notebooks 02 and 03). If you keep the data elsewhere, change the dataset paths below, or set environment variables with the same names before starting Jupyter.

In [ ]:
# ============================================================================ Global settings
import os
from pathlib import Path

# ---- 1. Master folder: every file the notebooks read or write lies below it (or set the environment variable ECGFM_RACE_DIR)
MASTER_DIR = Path(os.environ.get("ECGFM_RACE_DIR", "~/ECGFM_race_encoding_data")).expanduser()

DATA_DIR    = MASTER_DIR / "data"      # raw PhysioNet downloads (read only)
MODEL_DIR   = MASTER_DIR / "models"    # pretrained weights and the reference code they need (notebook 04)
WORK_DIR    = MASTER_DIR / "work"      # row-level intermediate files: cohorts, waveforms, embeddings, predictions (never share)
RESULTS_DIR = MASTER_DIR / "results"   # aggregate results written by notebooks 02-09 (counts, AUROCs, differences)
OUTPUT_DIR  = MASTER_DIR / "outputs"   # tables (csv) and figures (png, pdf) of the paper and its Supplementary Information (notebooks 10 and 11)

# ---- 2. Raw datasets. Defaults: the folders created by `wget -r -N -c -np https://physionet.org/files/<project>/<version>/`
#         when it is run inside DATA_DIR. If you unzipped the PhysioNet ZIP files instead, point each path at the unzipped folder.
PHYSIONET = DATA_DIR / "physionet.org" / "files"
ECHONEXT_DIR   = Path(os.environ.get("ECHONEXT_DIR",   PHYSIONET / "echonext" / "1.1.1"))       # EchoNext_metadata_100k.csv, EchoNext_<split>_waveforms.npy
MIMIC_IV_DIR   = Path(os.environ.get("MIMIC_IV_DIR",   PHYSIONET / "mimiciv" / "3.1"))          # hosp/admissions.csv.gz, hosp/patients.csv.gz
MIMIC_ED_DIR   = Path(os.environ.get("MIMIC_ED_DIR",   PHYSIONET / "mimic-iv-ed" / "2.2"))      # ed/edstays.csv.gz
MIMIC_ECG_DIR  = Path(os.environ.get("MIMIC_ECG_DIR",  PHYSIONET / "mimic-iv-ecg" / "1.0"))     # record_list.csv, machine_measurements.csv, files/
MIMIC_ECHO_DIR = Path(os.environ.get("MIMIC_ECHO_DIR", PHYSIONET / "mimic-iv-echo" / "1.0.1"))  # structured-measurement.csv.gz

# ---- 3. Compute
DEVICE = "cuda"              # GPU for embedding extraction and for most model fits ("cpu" works, but slowly)
GPU_MEMORY_FRACTION = 0.28   # cap on this process's share of GPU memory (None = no cap). The study kept every process below 1/3 of a shared 32 GB GPU.
CPU_JOBS = 10                # parallel workers for the scikit-learn fits of notebooks 07 and 08 (each worker uses 2 threads)

## 0.2 Fixed constants of the analysis

Do not change these values: they define the analysis reported in the paper (groups, diagnoses, models, folds, seeds and regularisation grids).

In [ ]:
# ============================================================================ Imports and fixed constants
import json
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as TF
from scipy.stats import rankdata
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)
for _d in (MODEL_DIR, WORK_DIR, RESULTS_DIR, OUTPUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---- groups
SPLITS = ["train", "val", "test", "no_split"]           # EchoNext release files; concatenated in this order they give the global row order
RACE4 = ["hispanic", "white", "black", "asian"]         # groups compared, coded 0..3 in this order (other and unknown = -1)
H, W, B, A = (RACE4.index(x) for x in ("hispanic", "white", "black", "asian"))
GROUPS6 = RACE4 + ["other", "unknown"]
GROUP_NAME = {"white": "White", "black": "Black", "hispanic": "Hispanic", "asian": "Asian", "other": "Other", "unknown": "Unknown", "All": "All patients"}
PAIRS = [("black", "white"), ("hispanic", "white"), ("asian", "white")]   # comparisons between groups (first minus second)

# ---- foundation models reported in the paper, and the same architectures with random weights
FMS = {"ecgfounder": "ECGFounder", "ecgfm": "ECG-FM", "hubert_large": "HuBERT-ECG"}
RANDS = {k + "_random": v + " (random weights)" for k, v in FMS.items()}
PAPER_ORDER = ["ecgfm", "ecgfounder", "hubert_large"]   # order of the models in the tables and figures

# ---- diagnoses. EchoNext: the release columns, in this order (the order fixes the bootstrap seed of each label)
E_LABELS = ["shd_moderate_or_greater_flag", "lvef_lte_45_flag", "lvwt_gte_13_flag", "aortic_stenosis_moderate_or_greater_flag",
            "aortic_regurgitation_moderate_or_greater_flag", "mitral_regurgitation_moderate_or_greater_flag",
            "tricuspid_regurgitation_moderate_or_greater_flag", "pulmonary_regurgitation_moderate_or_greater_flag",
            "rv_systolic_dysfunction_moderate_or_greater_flag", "pericardial_effusion_moderate_large_flag", "pasp_gte_45_flag", "tr_max_gte_32_flag"]
E_COMP, E_AS = "shd_moderate_or_greater_flag", "aortic_stenosis_moderate_or_greater_flag"
# MIMIC-IV: the 11 components derived in notebook 03; the labels of the analysis (composite + components that passed the ascertainment rule)
# are written to results/03_mimic/labels.json and read with mimic_labels()
M_COMPONENTS = ["lvef_le45", "lvwt_ge13", "as_modplus", "mr_modplus", "tr_modplus", "rv_modplus", "ph_modplus", "trv_ge32", "ar_modplus",
                "pr_modplus", "pe_modplus"]
M_COMP, M_AS = "shd_main", "as_modplus"
LABEL_NAME = {
    "shd_moderate_or_greater_flag": "Composite SHD", "lvef_lte_45_flag": "LVEF ≤45%", "lvwt_gte_13_flag": "LV wall thickness ≥1.3 cm",
    "aortic_stenosis_moderate_or_greater_flag": "Aortic stenosis", "aortic_regurgitation_moderate_or_greater_flag": "Aortic regurgitation",
    "mitral_regurgitation_moderate_or_greater_flag": "Mitral regurgitation", "tricuspid_regurgitation_moderate_or_greater_flag": "Tricuspid regurgitation",
    "pulmonary_regurgitation_moderate_or_greater_flag": "Pulmonary regurgitation", "rv_systolic_dysfunction_moderate_or_greater_flag": "RV systolic dysfunction",
    "pericardial_effusion_moderate_large_flag": "Pericardial effusion", "pasp_gte_45_flag": "PASP ≥45 mmHg", "tr_max_gte_32_flag": "TR velocity ≥3.2 m/s",
    "shd_main": "Composite SHD", "lvef_le45": "LVEF ≤45%", "lvwt_ge13": "LV wall thickness ≥1.3 cm", "as_modplus": "Aortic stenosis",
    "mr_modplus": "Mitral regurgitation", "tr_modplus": "Tricuspid regurgitation", "rv_modplus": "RV systolic dysfunction",
    "ph_modplus": "Pulmonary hypertension (reported severity)", "trv_ge32": "TR velocity ≥3.2 m/s", "ar_modplus": "Aortic regurgitation",
    "pr_modplus": "Pulmonary regurgitation", "pe_modplus": "Pericardial effusion"}

# ---- cross-fitting, bootstrap and probes
N_FOLDS, VAL_FRAC, SEED_FOLDS = 5, 0.15, 20261111      # 5 patient folds; 15% of the other patients -> validation (12% overall)
N_BOOT = 1000                                          # patient bootstrap resamples
SEED_PROBES, SEED_STRUCTURE, SEED_ERRORS, SEED_REFIT = 20261112, 20261109, 20261111, 20261112   # notebooks 05, 06, 07-08, 09
SENS = 0.80                                            # every detector's threshold gives 80% sensitivity among validation patients
MIN_N = 10                                             # a comparison needs at least 10 patients in each denominator
C_PROBE = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 1.0]   # race probes and structure models (notebooks 05, 06)
C_DETECTOR = [1e-4, 1e-3, 1e-2, 1e-1]                        # disease detectors (07, 09)
C_ERASED_DETECTOR = [1e-3, 1e-2, 1e-1]                       # disease detectors refitted after erasure (08)
C_ERASED_RACE = [1e-3, 1e-2, 1e-1, 1.0]                      # race probes refitted after erasure (08)

## 0.3 Files and data loaders

In [ ]:
def find_file(folder, name):
    """`folder/name`, matched without regard to upper or lower case (PhysioNet pages and file names differ in case for some files)."""
    p = Path(folder) / name
    if p.exists():
        return p
    hits = [q for q in Path(folder).glob("*") if q.name.lower() == name.lower()] if Path(folder).exists() else []
    if not hits:
        raise FileNotFoundError(f"{name} was not found in {folder}. See the download instructions in notebooks 02 and 03.")
    return hits[0]


def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=float), encoding="utf-8")


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def results(name):
    """Folder for the aggregate results of one notebook, e.g. results('05_race_probes')."""
    d = RESULTS_DIR / name
    d.mkdir(parents=True, exist_ok=True)
    return d


def work(*parts):
    """Folder for row-level intermediate files (never shared), e.g. work('echonext')."""
    d = WORK_DIR.joinpath(*parts)
    d.mkdir(parents=True, exist_ok=True)
    return d


def echonext_metadata():
    return pd.read_csv(find_file(ECHONEXT_DIR, "EchoNext_metadata_100k.csv"), low_memory=False)


def echonext_rows(meta=None):
    """All 100,000 EchoNext ECGs in the global row order: train, val, test and no_split, each in file order (= row order of its .npy file)."""
    meta = echonext_metadata() if meta is None else meta
    return pd.concat([meta[meta["split"] == s].reset_index(drop=True).assign(_split=s) for s in SPLITS], ignore_index=True)


def echonext_waveforms(split):
    """Memory-mapped N x 1 x 2500 x 12 array of one EchoNext split (250 Hz, 10 s, leads I, II, III, aVR, aVL, aVF, V1-V6)."""
    return np.load(find_file(ECHONEXT_DIR, f"EchoNext_{split}_waveforms.npy"), mmap_mode="r")


def emb_echonext(name):
    """Embeddings of the 100,000 EchoNext ECGs (float32, global row order) from encoder `name` (notebook 04)."""
    return np.concatenate([np.load(WORK_DIR / "embeddings" / "echonext" / name / f"{s}.npy") for s in SPLITS])


def emb_mimic(name):
    """Embeddings of every candidate MIMIC-IV ECG (float32); index the rows with the cohort column `row`."""
    return np.load(WORK_DIR / "embeddings" / "mimic" / name / "all.npy")


def load_design(cohort):
    """Patient-level 5-fold design written by notebook 02 ('echonext') or 03 ('mimic')."""
    return dict(np.load(WORK_DIR / cohort / "design.npz", allow_pickle=True))


def fold_rows(D, k):
    """Rows of round k: fitting = every ECG of the fitting patients; validation and evaluation = the most recent ECG of each patient."""
    r = D["role"][k][D["pi_of_row"]]
    return np.flatnonzero(r == 0), np.flatnonzero((r == 1) & D["mr_row"]), np.flatnonzero((r == 2) & D["mr_row"])


def load_mimic_cohort():
    """Final MIMIC-IV cohort: candidate ECGs of notebook 03 whose waveform passed the check. `row` indexes the waveform and embedding arrays,
    `mr` marks the most recent ECG of each patient."""
    C = pd.read_parquet(WORK_DIR / "mimic" / "cohort_candidates.parquet")
    qc = pd.read_parquet(WORK_DIR / "mimic" / "waveform_qc.parquet")
    assert len(C) == len(qc) and (C.row.to_numpy() == qc.row.to_numpy()).all()
    C = C[qc.wave_ok.to_numpy()].copy()
    last = C.sort_values(["t_ecg", "study_id"]).drop_duplicates("subject_id", keep="last").index
    C["mr"] = False
    C.loc[last, "mr"] = True
    return C.reset_index(drop=True)


def mimic_labels():
    """MIMIC-IV labels of the analysis: the composite first, then the components that passed the ascertainment rule (notebook 03)."""
    return load_json(RESULTS_DIR / "03_mimic" / "labels.json")["main"]


def race_index(values):
    """Race and ethnicity category -> 0..3 (RACE4 order); other and unknown -> -1."""
    return pd.Series(values).map({r: i for i, r in enumerate(RACE4)}).fillna(-1).to_numpy(int)


def strata6(age, male):
    """Six age bands (<40, 40-49, 50-59, 60-69, 70-79, >=80) x sex = 12 strata, used for direct standardisation."""
    return np.digitize(age, [40, 50, 60, 70, 80]) * 2 + np.asarray(male, int)


def median_iqr(v, d=0):
    v = pd.Series(v).dropna()
    return f"{v.median():.{d}f} ({v.quantile(.25):.{d}f}–{v.quantile(.75):.{d}f})"


def prevalence_rows(cohort, labels, Y_lat, g_lat, Y_all, g_all):
    """Prevalence of every label in every group: on the most recent ECG of each patient and over all ECGs (the rows from which probes and
    erasers are fitted)."""
    rows = []
    for j, lab in enumerate(labels):
        ok = np.isfinite(Y_lat[:, j]) & (g_lat >= 0)
        pos = np.bincount(g_lat[ok], weights=Y_lat[ok, j], minlength=4)
        nn = np.bincount(g_lat[ok], minlength=4)
        for gi, gn in enumerate(RACE4):
            rows.append({"cohort": cohort, "label_key": lab, "label": LABEL_NAME[lab], "group": gn, "n_latest": int(nn[gi]),
                         "pos_latest": int(pos[gi]), "prev_latest": pos[gi] / nn[gi],
                         "prev_all_ecgs": float(np.nanmean(Y_all[g_all == gi, j])), "n_all_ecgs": int((g_all == gi).sum())})
    return rows

## 0.4 AUROC, bootstrap and confidence intervals

In [ ]:
def fast_auc(y, s):
    """AUROC from ranks (ties get average ranks); NaN if one class is missing."""
    y = np.asarray(y, bool)
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((rankdata(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def boot_idx(n, seed, n_boot=N_BOOT):
    """Patient bootstrap resamples (indices); the same seed gives the same resamples, so comparisons are paired."""
    rng = np.random.default_rng(seed)
    return [rng.integers(0, n, n) for _ in range(n_boot)]


def ci95(v):
    """Percentile 95% confidence interval of bootstrap values."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 0.5 L2-regularised logistic regression

The probes and detectors are L2-regularised logistic regressions on standardised embeddings, fitted with one of two numerically equivalent solvers:

* **GPU solver** (`lr_fit`, `lr_select`): full-batch L-BFGS in float64, with the loss and gradient computed on the GPU. It minimises the same objective as scikit-learn, `mean(loss) + ||w||² / (2·C·n)` with an unpenalised intercept, and stops at the same gradient tolerance. Used for MIMIC-IV, for every race probe and structure model, for the aortic-stenosis-negative erasure and for the prevalence-equalised refit.
* **CPU solver** (`lr_select_cpu`): scikit-learn `LogisticRegression(solver="lbfgs")`, warm-started along the grid of C. Used for the EchoNext disease detectors (notebook 07) and the EchoNext standard and SHD-negative erasure (notebook 08).

The problem is convex, so the two solvers reach the same solution up to the stopping tolerance. The regularisation strength C is chosen on validation patients (AUROC, or the mean one-versus-rest AUROC for the four-group probe).

In [ ]:
DTYPE = torch.float64
_DEVICE = None


def device():
    """CUDA device with the memory cap of the global settings (set once per process), or the CPU."""
    global _DEVICE
    if _DEVICE is None:
        if DEVICE == "cuda" and torch.cuda.is_available():
            if GPU_MEMORY_FRACTION:
                torch.cuda.set_per_process_memory_fraction(GPU_MEMORY_FRACTION, 0)
            _DEVICE = torch.device("cuda")
        else:
            _DEVICE = torch.device("cpu")
    return _DEVICE


def free_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def to_dev(X):
    return torch.as_tensor(np.ascontiguousarray(X), dtype=DTYPE, device=device())


def col_mean_var(X, chunk=8192):
    """Column mean and population variance in row chunks (a large float64 tensor is never copied as a whole)."""
    n = X.shape[0]
    mu = torch.zeros((1, X.shape[1]), dtype=X.dtype, device=X.device)
    for i in range(0, n, chunk):
        mu += X[i:i + chunk].sum(0, keepdim=True)
    mu /= n
    var = torch.zeros_like(mu)
    for i in range(0, n, chunk):
        var += ((X[i:i + chunk] - mu) ** 2).sum(0, keepdim=True)
    return mu, var / n


def standardize_(Xtr, *others, chunk=8192):
    """In place: subtract the mean and divide by the SD of the fitting rows (population SD, like StandardScaler; constant columns unscaled)."""
    mu, var = col_mean_var(Xtr, chunk)
    sd = var.sqrt()
    sd = torch.where(sd == 0, torch.ones_like(sd), sd)
    for X in (Xtr,) + others:
        for i in range(0, X.shape[0], chunk):
            X[i:i + chunk] = (X[i:i + chunk] - mu) / sd
    return mu, sd


@torch.no_grad()
def lr_fit(X, y, C, k=None, w0=None, max_iter=3000, tol=1e-4):
    """GPU solver. X: n x d tensor; y: 0/1 tensor (binary) or class index with k classes (multinomial).
    -> ((W, b, n_iter), w_flat); w_flat warm-starts the next fit along the C grid."""
    from scipy.optimize import minimize
    n, d = X.shape
    multi = k is not None
    shape = (k, d + 1) if multi else (d + 1,)
    lam = 1.0 / (C * n)
    if multi:
        Y = torch.zeros((n, k), dtype=DTYPE, device=X.device)
        Y[torch.arange(n, device=X.device), y.long()] = 1
    else:
        Y = y.to(DTYPE)

    def fg(w_np):
        w = torch.from_numpy(w_np).to(X.device).view(shape)
        if multi:
            z = X @ w[:, :d].T + w[:, d]
            loss = (torch.logsumexp(z, 1) - (z * Y).sum(1)).mean() + 0.5 * lam * (w[:, :d] ** 2).sum()
            r = (torch.softmax(z, 1) - Y) / n
            g = torch.cat([r.T @ X + lam * w[:, :d], r.sum(0)[:, None]], 1)
        else:
            z = X @ w[:d] + w[d]
            loss = (TF.softplus(z) - Y * z).mean() + 0.5 * lam * (w[:d] ** 2).sum()
            r = (torch.sigmoid(z) - Y) / n
            g = torch.cat([X.T @ r + lam * w[:d], r.sum()[None]])
        return float(loss), g.reshape(-1).cpu().numpy()

    x0 = np.zeros(int(np.prod(shape))) if w0 is None else np.asarray(w0, np.float64)
    res = minimize(fg, x0, method="L-BFGS-B", jac=True, options={"maxiter": max_iter, "maxls": 50, "gtol": tol, "ftol": 64 * np.finfo(float).eps})
    w = torch.from_numpy(res.x).to(X.device).view(shape)
    return ((w[:, :d], w[:, d], int(res.nit)) if multi else (w[:d], w[d], int(res.nit))), res.x


def lr_scores(X, W, b):
    """Binary: logits (n,); multinomial: class probabilities (n, k). Returned as numpy float64."""
    z = X @ W.T + b if W.dim() == 2 else X @ W + b
    return (torch.softmax(z, 1) if W.dim() == 2 else z).cpu().numpy()


def auc(y, s):
    return float(roc_auc_score(y, s))


def macro_ovr(y, P):
    """One-versus-rest AUROC of every class."""
    return [auc(y == j, P[:, j]) for j in range(P.shape[1])]


def lr_select(Xtr, ytr, Xva, yva, grid, k=None):
    """GPU solver along the C grid (warm-started); C chosen by validation AUROC (mean one-versus-rest AUROC if multinomial).
    ytr, yva: numpy arrays. -> dict(C, val, W, b)."""
    yt = torch.as_tensor(ytr, device=Xtr.device)
    best, w0 = None, None
    for C in grid:
        (Wc, bc, _), w0 = lr_fit(Xtr, yt, C, k=k, w0=w0)
        sv = lr_scores(Xva, Wc, bc)
        a = float(np.mean(macro_ovr(yva, sv))) if k else auc(yva, sv)
        if best is None or a > best["val"]:
            best = {"C": C, "val": a, "W": Wc, "b": bc}
    return best


def lr_select_cpu(Xtr, ytr, Xva, yva, grid, multiclass=False):
    """CPU solver (scikit-learn lbfgs, warm-started along the C grid); C chosen by validation AUROC (mean one-versus-rest AUROC if multiclass).
    -> (validation AUROC, C, coefficients, intercepts)."""
    clf = LogisticRegression(C=grid[0], max_iter=3000, tol=1e-4, warm_start=True)
    best = None
    for C in grid:
        clf.set_params(C=C)
        clf.fit(Xtr, ytr)
        pv = clf.predict_proba(Xva)
        a = float(np.mean([roc_auc_score(yva == j, pv[:, j]) for j in range(pv.shape[1])])) if multiclass else roc_auc_score(yva, pv[:, 1])
        if best is None or a > best[0]:
            best = (a, C, clf.coef_.copy(), clf.intercept_.copy())
    return best

## 0.6 Cross-fitted race probes

`probe_folds` fits, in each of the five rounds, a four-group probe (multinomial) and a Black-versus-White probe on the fitting rows, chooses C on the validation patients and scores the evaluation patients. Every patient is scored once, by probes that never saw any of that patient's ECGs. `evaluate` pools the out-of-fold scores and adds patient-bootstrap confidence intervals.

In [ ]:
def impute(Xtr, others):
    """Median of the fitting rows for missing values, plus a missing indicator for every column with missing values in the fitting rows."""
    med = np.nanmedian(Xtr, axis=0)
    cols = np.flatnonzero(np.isnan(Xtr).any(0))
    f = lambda Z: np.hstack([np.where(np.isnan(Z), med, Z), np.isnan(Z[:, cols]).astype(float)])
    return f(Xtr), [f(Z) for Z in others]


def probe_folds(Xall, D, g, row_mask=None, transform=None, do_impute=False, tasks=("4", "bw"), grid=C_PROBE):
    """Xall: features of every ECG (rows as in the design); g: group index of every ECG (-1 = not one of the four groups).
    -> P4 (patients x 4 probabilities), SB (patients, Black-versus-White logit), per-round info. Patients not scored keep NaN."""
    n_pat = len(D["fold_of"])
    P4, SB, info = np.full((n_pat, 4), np.nan), np.full(n_pat, np.nan), []
    keep = (g >= 0) if row_mask is None else ((g >= 0) & row_mask)
    for k in range(N_FOLDS):
        tr, va, ev = (r[keep[r]] for r in fold_rows(D, k))
        Z, (Zv, Ze) = impute(Xall[tr], [Xall[va], Xall[ev]]) if do_impute else (Xall[tr], [Xall[va], Xall[ev]])
        Xtr, Xva, Xev = (to_dev(z) for z in (Z, Zv, Ze))
        extra = {}
        if transform is not None:
            Xtr, Xva, Xev, extra = transform(k, Xtr, Xva, Xev, tr, va, ev)
        standardize_(Xtr, Xva, Xev)
        rec = {"fold": k, "n_fit": int(len(tr)), "n_val": int(len(va)), "n_eval": int(len(ev)), **extra}
        pe = D["pi_of_row"][ev]
        if "4" in tasks:
            b4 = lr_select(Xtr, g[tr], Xva, g[va], grid, k=4)
            P4[pe] = lr_scores(Xev, b4["W"], b4["b"])
            rec.update(C_4=b4["C"], val_macro=b4["val"])
        if "bw" in tasks:
            mtr, mva = np.isin(g[tr], [B, W]), np.isin(g[va], [B, W])
            bb = lr_select(Xtr[torch.as_tensor(mtr, device=device())], (g[tr][mtr] == B).astype(int),
                           Xva[torch.as_tensor(mva, device=device())], (g[va][mva] == B).astype(int), grid)
            SB[pe] = lr_scores(Xev, bb["W"], bb["b"])
            rec.update(C_bw=bb["C"], val_bw=bb["val"])
        info.append(rec)
        del Xtr, Xva, Xev
        free_gpu()
    return P4, SB, info


def macro4(y, P):
    return float(np.mean([fast_auc(y == j, P[:, j]) for j in range(4)]))


def evaluate(P4, SB, g_lat, seed, subset=None, boots=None):
    """Pooled out-of-fold AUROCs over the scored patients (optionally a subset), with patient-bootstrap 95% CIs.
    -> (record, {'bw': bootstrap values, 'macro': bootstrap values})."""
    m = np.isfinite(SB) & (g_lat >= 0) if subset is None else np.isfinite(SB) & (g_lat >= 0) & subset
    idx = np.flatnonzero(m)
    y, s, P = g_lat[idx], SB[idx], P4[idx]
    bw = np.isin(y, [B, W])
    rec = {"n_4group": int(len(idx)), "n_black": int((y == B).sum()), "n_white": int((y == W).sum()), "bw_auroc": fast_auc(y[bw] == B, s[bw])}
    has4 = np.isfinite(P).all()
    if has4:
        for j, gn in enumerate(RACE4):
            rec[f"ovr_{gn}"] = fast_auc(y == j, P[:, j])
        rec["macro_auroc"] = float(np.mean([rec[f"ovr_{gn}"] for gn in RACE4]))
    bs = {"bw": [], "macro": []}
    for ix in (boots if boots is not None else boot_idx(len(idx), seed)):
        yy = y[ix]; b2 = np.isin(yy, [B, W])
        bs["bw"].append(fast_auc(yy[b2] == B, s[ix][b2]))
        bs["macro"].append(macro4(yy, P[ix]) if has4 else np.nan)
    bs = {key: np.array(v) for key, v in bs.items()}
    rec["bw_lo"], rec["bw_hi"] = ci95(bs["bw"])
    if has4:
        rec["macro_lo"], rec["macro_hi"] = ci95(bs["macro"])
    return rec, bs

## 0.7 Error measures at the common threshold

For a detector with one threshold for every group, each group has a sensitivity and a false-positive rate (FPR). Two summary measures describe how a group differs from White patients:

* **flagging rate** = (sensitivity + FPR) / 2: how often the group is flagged, averaged over patients with and without the diagnosis (independent of prevalence). A difference means a shifted operating point.
* **balanced accuracy** = (sensitivity + specificity) / 2: how well patients with and without the diagnosis are separated at the threshold.

Differences between groups are directly standardised for age and sex: rates within the 12 strata of six age bands × sex, averaged with the same weights for both groups (the pooled distribution of the two groups among patients with the diagnosis for sensitivity, without it for the FPR). With FNR = 1 − sensitivity: ΔFLAG = (ΔFPR − ΔFNR)/2 and ΔBA = −(ΔFNR + ΔFPR)/2.

In [ ]:
K_STRATA = 12
MEASURES = ["FNR", "FPR", "FLAG", "BA"]


def cell_counts(y, pr, g, s, K=K_STRATA):
    """Patients with the diagnosis (P), missed (FN), without (N) and flagged without (FP), per group x stratum."""
    c = g * K + s
    pos = y == 1
    P = np.bincount(c[pos], minlength=4 * K).reshape(4, K)
    FN = np.bincount(c[pos & (pr == 0)], minlength=4 * K).reshape(4, K)
    N = np.bincount(c[~pos], minlength=4 * K).reshape(4, K)
    FP = np.bincount(c[~pos & (pr == 1)], minlength=4 * K).reshape(4, K)
    return P, FN, N, FP


def std_diff(num, den, a, b):
    """Directly standardised difference of a rate between groups a and b (weights: pooled stratum sizes of the two groups)."""
    ok = (den[a] > 0) & (den[b] > 0)
    if not ok.any():
        return np.nan
    w = (den[a] + den[b])[ok].astype(float)
    w /= w.sum()
    return float((num[a][ok] / den[a][ok]) @ w - (num[b][ok] / den[b][ok]) @ w)


def error_differences(y, pr, g, s):
    """-> d[pair, measure (FNR, FPR, FLAG, BA)], standardised for age and sex, and patients with and without the diagnosis per group."""
    P, FN, N, FP = cell_counts(y, pr, g, s)
    d = np.full((len(PAIRS), 4), np.nan)
    for i, (a, b) in enumerate(PAIRS):
        a, b = RACE4.index(a), RACE4.index(b)
        d[i, 0], d[i, 1] = std_diff(FN, P, a, b), std_diff(FP, N, a, b)
    d[:, 2] = (d[:, 1] - d[:, 0]) / 2
    d[:, 3] = -(d[:, 0] + d[:, 1]) / 2
    return d, P.sum(1), N.sum(1)


def difference_table(cohort, labels, Y, g, s, pred):
    """Differences between groups for every label, model and eraser, with 1000 patient-bootstrap resamples per label.
    labels: label keys in the order that fixes the seeds (SEED_ERRORS + label index); Y: patients x labels (most recent ECG);
    pred[model][eraser][label] = 0/1 prediction of every patient ('none' = the detector without erasure).
    The resamples are the same for every model and eraser, so changes after erasure are paired."""
    rows = []
    sel = g >= 0
    gs, ss = g[sel], s[sel]
    fms = list(pred)
    for j, lab in enumerate(labels):
        erasers = [e for e in pred[fms[0]] if lab in pred[fms[0]][e]]
        if not erasers:
            continue
        y = Y[sel, j]
        P = {fm: {e: np.asarray(pred[fm][e][lab])[sel] for e in erasers} for fm in fms}
        point = {fm: {e: error_differences(y, P[fm][e], gs, ss) for e in erasers} for fm in fms}
        boot = {fm: {e: np.empty((N_BOOT, len(PAIRS), 4)) for e in erasers} for fm in fms}
        rng = np.random.default_rng(SEED_ERRORS + j)
        for b in range(N_BOOT):
            ix = rng.integers(0, len(y), len(y))
            yb, gb, sb = y[ix], gs[ix], ss[ix]
            for fm in fms:
                for e in erasers:
                    boot[fm][e][b] = error_differences(yb, P[fm][e][ix], gb, sb)[0]
        for fm in fms:
            _, Pt, Nt = point[fm]["none"]
            d0, bd0 = point[fm]["none"][0], boot[fm]["none"]
            for e in erasers:
                d, bd = point[fm][e][0], boot[fm][e]
                for i, (a, b_) in enumerate(PAIRS):
                    ai, bi = RACE4.index(a), RACE4.index(b_)
                    ok_pos, ok_neg = min(Pt[ai], Pt[bi]) >= MIN_N, min(Nt[ai], Nt[bi]) >= MIN_N
                    for mi, meas in enumerate(MEASURES):
                        valid = {"FNR": ok_pos, "FPR": ok_neg}.get(meas, ok_pos and ok_neg)
                        lo, hi = ci95(bd[:, i, mi])
                        r = {"cohort": cohort, "fm": FMS[fm], "fm_key": fm, "label_key": lab, "label": LABEL_NAME.get(lab, lab), "pair": f"{a}-{b_}",
                             "eraser": e, "measure": meas, "valid": bool(valid), "pos_a": int(Pt[ai]), "pos_b": int(Pt[bi]),
                             "neg_a": int(Nt[ai]), "neg_b": int(Nt[bi]), "value": d[i, mi], "lo": lo, "hi": hi}
                        if e != "none":
                            clo, chi = ci95(bd[:, i, mi] - bd0[:, i, mi])
                            r.update(before=d0[i, mi], change=d[i, mi] - d0[i, mi], change_lo=clo, change_hi=chi)
                        rows.append(r)
    return pd.DataFrame(rows)

## 0.8 Least-squares concept erasure (LEACE)

LEACE (Belrose et al., NeurIPS 2023) computes in closed form the smallest change to the embeddings after which no linear classifier can predict group membership: it removes the directions along which the group means differ, after whitening. It is estimated from the standardised fitting ECGs of the four groups and applied to every ECG. `leace_fit_np` (NumPy, used with the CPU solver) and `leace_fit_torch` (PyTorch, used with the GPU solver) implement the same algebra.

In [ ]:
def onehot(g, k=4):
    Z = np.zeros((len(g), k)); Z[np.arange(len(g)), g] = 1
    return Z


def leace_fit_np(X, Z):
    """X: n x d standardised embeddings, Z: n x 4 one-hot groups. -> (mean, projection M, rank removed); erased X = X - (X - mean) @ M.T"""
    mu = X.mean(0)
    Xc, Zc = X - mu, Z - Z.mean(0)
    n = len(X)
    Sxx, Sxz = Xc.T @ Xc / n, Xc.T @ Zc / n
    ev, V = np.linalg.eigh(Sxx)
    ok = ev > ev.max() * 1e-10
    Wh = (V[:, ok] * ev[ok] ** -0.5) @ V[:, ok].T                      # whitening
    Wp = (V[:, ok] * ev[ok] ** 0.5) @ V[:, ok].T                       # its pseudo-inverse
    U, s, _ = np.linalg.svd(Wh @ Sxz, full_matrices=False)
    U = U[:, s > s.max() * 1e-8]                                       # directions of the whitened cross-covariance
    return mu, Wp @ (U @ U.T) @ Wh, int(U.shape[1])


def leace_fit_torch(X, g):
    """Same algebra on the GPU. X: n x d tensor, g: n group indices (numpy). -> (mean, M, rank)."""
    n, d = X.shape
    Z = torch.zeros((n, 4), dtype=DTYPE, device=X.device)
    Z[torch.arange(n, device=X.device), torch.as_tensor(g, device=X.device)] = 1
    mu = col_mean_var(X)[0].ravel()
    Zc = Z - Z.mean(0)
    Sxx = X.T @ X / n - torch.outer(mu, mu)
    Sxz = X.T @ Zc / n
    ev, V = torch.linalg.eigh(Sxx)
    ok = ev > ev.max() * 1e-10
    Wh = (V[:, ok] * ev[ok] ** -0.5) @ V[:, ok].T
    Wp = (V[:, ok] * ev[ok] ** 0.5) @ V[:, ok].T
    U, s, _ = torch.linalg.svd(Wh @ Sxz, full_matrices=False)
    U = U[:, s > s.max() * 1e-8]
    return mu, Wp @ (U @ U.T) @ Wh, int(U.shape[1])


def leace_apply_torch(X, e):
    """Erased copy of X (in row chunks): X - (X - mean) @ M.T"""
    out = torch.empty_like(X)
    for i in range(0, len(X), 40000):
        out[i:i + 40000] = X[i:i + 40000] - (X[i:i + 40000] - e[0]) @ e[1].T
    return out

## 0.9 Matching Black and White patients on cardiac structure

Each Black patient is paired with a White patient of the same sex and clinical setting of the ECG by optimal one-to-one assignment on the standardised Euclidean distance in age, LVEF, the two wall thicknesses and calendar year. Calipers on that distance are tried from none to tight, and the loosest caliper that brings every standardised mean difference below 0.1 is kept. The within-pair concordance (the share of pairs in which the Black patient has the higher probe score) is a matched-pair AUROC.

In [ ]:
MATCH_COVS = ["age", "lvef", "ivs", "lvpw", "year"]
CALIPERS = [np.inf, 2.0, 1.5, 1.0, 0.75, 0.5, 0.35]


def match(Dt, ga, gb):
    """Dt: per-patient arrays (keep, core, y4, age, lvef, ivs, lvpw, year, male, loc). -> (pairs, balance table, info)."""
    from scipy.optimize import linear_sum_assignment
    elig = Dt["keep"] & Dt["core"]
    ia, ib = np.flatnonzero(elig & (Dt["y4"] == ga)), np.flatnonzero(elig & (Dt["y4"] == gb))
    pool = np.concatenate([ia, ib])
    Xc = np.column_stack([Dt[c] for c in MATCH_COVS])
    Zc = (Xc - Xc[pool].mean(0)) / Xc[pool].std(0)
    pairs, dist = [], []                                               # optimal assignment once (no caliper), then drop pairs above each caliper
    for sex in (0.0, 1.0):
        for loc in np.unique(Dt["loc"][pool]):
            a = ia[(Dt["male"][ia] == sex) & (Dt["loc"][ia] == loc)]
            b = ib[(Dt["male"][ib] == sex) & (Dt["loc"][ib] == loc)]
            if len(a) and len(b):
                cost = ((Zc[a][:, None, :] - Zc[b][None, :, :]) ** 2).sum(-1)
                ra, rb = linear_sum_assignment(cost)
                pairs += list(zip(a[ra], b[rb])); dist += list(np.sqrt(cost[ra, rb]))
    pairs, dist = np.array(pairs), np.array(dist)

    def balance(pp):
        rows = []
        for c in MATCH_COVS + ["male"]:
            for tag, xa, xb in (("before", Dt[c][ia], Dt[c][ib]), ("after", Dt[c][pp[:, 0]], Dt[c][pp[:, 1]])):
                rows.append({"covariate": c, "when": tag, "mean_a": xa.mean(), "mean_b": xb.mean(),
                             "smd": (xa.mean() - xb.mean()) / np.sqrt((xa.var() + xb.var()) / 2 + 1e-12)})
        return pd.DataFrame(rows)

    for cal in CALIPERS:
        pp = pairs[dist <= cal]
        bal = balance(pp)
        if (bal[bal.when == "after"].smd.abs() < 0.1).all():
            break
    info = {"caliper_sd_distance": None if np.isinf(cal) else cal, "n_pairs": int(len(pp)), "n_a_eligible": int(len(ia)),
            "n_b_eligible": int(len(ib)), "max_abs_smd_after": float(bal[bal.when == "after"].smd.abs().max())}
    return pp, bal, info


def concordance(pp, scores, seed=SEED_STRUCTURE + 7):
    """Within-pair concordance of each score (ties count one half), with a pair-bootstrap 95% CI."""
    rng = np.random.default_rng(seed)
    boots = [rng.integers(0, len(pp), len(pp)) for _ in range(N_BOOT)]
    rows = []
    for name, s in scores.items():
        sa, sb = s[pp[:, 0]], s[pp[:, 1]]
        conc = (sa > sb).astype(float) + 0.5 * (sa == sb)
        bs = [conc[ix].mean() for ix in boots]
        rows.append({"score": name, "pair_concordance": conc.mean(), "lo": np.percentile(bs, 2.5), "hi": np.percentile(bs, 97.5)})
    return pd.DataFrame(rows)

## 0.10 Plot style (notebooks 10 and 11)

Colour follows the cohort (EchoNext blue, MIMIC-IV orange) and comparators are grey; marker shape and fill carry the second distinction, so that no figure relies on colour alone.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import MultipleLocator

plt.rcParams.update({"font.family": ["Arial", "Liberation Sans", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": True, "pdf.fonttype": 42,
                     "svg.fonttype": "none", "axes.linewidth": 0.8})
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)         # quiet if Arial is not installed (a similar font is used)
CE, CM = "#2a78d6", "#eb6834"                                              # EchoNext, MIMIC-IV
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
COH = {"EchoNext": CE, "MIMIC-IV": CM}


def style(ax, grid="x"):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(AXIS); ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=7.5, length=0)
    if grid:
        (ax.xaxis if grid == "x" else ax.yaxis).grid(True, color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


def separators(ax, n):
    """Light horizontal lines between the rows of a forest-type panel (rows at y = 0 .. n-1)."""
    for i in range(n - 1):
        ax.axhline(i + 0.5, color=GRID, lw=0.6, zorder=0)


def panel(ax, letter, title=None):
    """Panel header: bold lower-case letter at the left edge of the axes, title right after it."""
    kw = dict(xy=(0, 1), xycoords="axes fraction", textcoords="offset points", color=INK, va="bottom", ha="left", fontweight="bold", annotation_clip=False)
    if letter:
        ax.annotate(letter, xytext=(0, 6), fontsize=10.5, **kw)
    if title:
        ax.annotate(title, xytext=(13 if letter else 0, 6.8), fontsize=8.6, **kw)


def hints(ax, left, right):
    """Small reading aids in the top corners of a difference axis."""
    if left:
        ax.text(0.012, 0.985, "← " + left, transform=ax.transAxes, fontsize=6.6, color=MUTED, va="top", ha="left")
    if right:
        ax.text(0.988, 0.985, right + " →", transform=ax.transAxes, fontsize=6.6, color=MUTED, va="top", ha="right")


def save_figure(fig, name):
    """Save a figure as PNG (400 dpi) and PDF under OUTPUT_DIR/figures and show it in the notebook."""
    d = OUTPUT_DIR / "figures"
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f"{name}.png", dpi=400, facecolor="white", bbox_inches="tight")
    fig.savefig(d / f"{name}.pdf", facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 0.11 Number formatting of the tables (notebooks 10 and 11)

Numbers are printed as in the paper: true minus signs (−), differences with an explicit sign, AUROCs with three decimals.

In [ ]:
MINUS = "−"


def signed(v, d=1):
    """Signed number with a true minus sign; a value that rounds to zero is printed without a sign."""
    s = f"{v:+.{d}f}"
    return s[1:] if float(s) == 0 else s.replace("-", MINUS)


def pp(lo, hi, d=1):
    """Range of two proportions as percentage points, e.g. '+2.7 to +5.6'."""
    return f"{signed(100 * lo, d)} to {signed(100 * hi, d)}"


def rngd(lo, hi, d=2):
    return f"{lo:.{d}f}–{hi:.{d}f}"


f3 = lambda x: "" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.3f}"
ci3 = lambda v, lo, hi: f"{v:.3f} ({lo:.3f}–{hi:.3f})"
mn = lambda s: s.replace("-", MINUS)


def pfmt(p):
    """P value as printed in the figures: '= 0.001', '= 0.10' or '< 0.001'."""
    return "< 0.001" if p < 0.0005 else (f"= {p:.3f}" if p < 0.01 else f"= {p:.2f}")


def save_table(df, name):
    """Save a display table as CSV under OUTPUT_DIR/tables (UTF-8 with BOM, so that spreadsheet programs show the symbols) and return it."""
    d = OUTPUT_DIR / "tables"
    d.mkdir(parents=True, exist_ok=True)
    df.to_csv(d / f"{name}.csv", index=False, encoding="utf-8-sig")
    return df

## 0.12 Aggregate results for the tables and figures (notebooks 10 and 11)

`load_all()` reads every aggregate result that the tables and figures need (from `RESULTS_DIR` only, never row-level files). `tmq()` selects the differences between groups for one cohort, diagnosis, measure, comparison and eraser: one row per model, in the order of the paper.

In [ ]:
ORDER = [FMS[k] for k in PAPER_ORDER]          # model names in the order of the paper: ECG-FM, ECGFounder, HuBERT-ECG
KEY = {FMS[k]: k for k in FMS}                 # display name -> model key
N_FM, N_WORD = len(ORDER), "three"


def read_result(nb, name):
    return pd.read_csv(RESULTS_DIR / nb / name)


def load_all():
    """Every aggregate result the tables and figures need."""
    R = {"s1e": read_result("05_race_probes", "echonext_probe_results.csv").set_index("model"),
         "s1m": read_result("05_race_probes", "mimic_probe_results.csv").set_index("model"),
         "s1ec": read_result("05_race_probes", "echonext_comparisons.csv"), "s1mc": read_result("05_race_probes", "mimic_comparisons.csv")}
    for c, k in (("echonext", "e2"), ("mimic", "m2")):
        R[k] = {"m0": read_result("06_cardiac_structure", f"{c}_structure_only.csv").set_index("model"),
                "A": read_result("06_cardiac_structure", f"{c}_no_recorded_shd.csv").set_index("model"),
                "B": read_result("06_cardiac_structure", f"{c}_matched_concordance.csv").set_index("score"),
                "C": read_result("06_cardiac_structure", f"{c}_structure_removed.csv"),
                "sum": load_json(RESULTS_DIR / "06_cardiac_structure" / f"{c}_summary.json")}
    R["disc"] = read_result("07_disease_detectors", "discrimination.csv")
    before = read_result("07_disease_detectors", "differences_before_erasure.csv")   # detectors without erasure (notebook 07)
    after = read_result("08_race_erasure", "differences.csv")                          # after erasure, with paired changes (notebook 08)
    R["tm"] = pd.concat([before, after[after.eraser != "none"]], ignore_index=True)
    R["race_after"] = read_result("08_race_erasure", "race_after_erasure.csv")
    R["dauc"] = read_result("08_race_erasure", "detector_auroc.csv")
    R["ch"] = pd.concat([read_result("02_echonext", "characteristics.csv"), read_result("03_mimic", "characteristics.csv")], ignore_index=True)
    R["prev"] = pd.concat([read_result("02_echonext", "prevalence.csv"), read_result("03_mimic", "prevalence.csv")], ignore_index=True)
    R["x2"] = read_result("09_prevalence_equalised_refit", "summary.csv")
    R["x2rows"] = read_result("09_prevalence_equalised_refit", "fitting_rows.csv")
    return R


def tmq(R, cohort, label, measure, pair="black-white", eraser="none"):
    """Rows of the difference table for one cohort, diagnosis, measure, comparison and eraser: one row per model, in the order of the paper."""
    d = R["tm"]
    d = d[(d.cohort == cohort) & (d.label_key == label) & (d.measure == measure) & (d.pair == pair) & (d.eraser == eraser) & d.valid].set_index("fm")
    return d.loc[[f for f in ORDER if f in d.index]]

print(f"Settings loaded. MASTER_DIR = {MASTER_DIR}")